# Logan database variants: acquisition and build

This notebook performs an actual build using six public Logan contig assemblies from `s3://logan-pub` and matching run metadata from the ENA API. The upstream Logan project does not publish database modes named `metadata` and `no_metadata`. For this controlled benchmark, `no_metadata` stores sequence identifiers, sequences, lengths, and GC fractions. `metadata` stores the identical sequence payload plus normalized ENA run metadata, Logan abundance values, and supporting indexes.

The experiment alternates build order over three repetitions. It records provenance, source checksums, build time, insertion throughput, database size, SQLite integrity checks, and a SHA-256 digest proving that both variants contain identical sequence identifiers and sequence data.

In [1]:
from pathlib import Path
from IPython.display import display
from benchmark_core import benchmark_environment, prepare_and_build, project_root

ROOT = project_root()
ROOT

PosixPath('/home/runner/workspace/logan-database-benchmarks')

In [2]:
environment = benchmark_environment()
display(environment.T)

,0
timestamp_utc,2026-09-08T03:11:34.098337+00:00
python,3.11.14
sqlite,3.51.1
platform,Linux-6.18.49-x86_64-with-glibc2.40
cpu_logical,4
cpu_physical,4
memory_gib,7.778587
logan_revision,b3da25311802544c4c0e53c44b98380acf4d04d6
logan_analysis_revision,5e92dafb87e40834ff2123d9b55e3bb27887fedb
petadex_revision,a6bc326eb187925591b3988f57b4fadb1dba557d


In [3]:
downloads, metadata, sample_summary, builds, validation = prepare_and_build(ROOT)
display(sample_summary)
display(downloads[['url', 'bytes', 'decompressed_bytes', 'sha256', 'etag']])

,accessions,sequences,total_bases,compressed_mib,decompressed_mib,metadata_rows
0,6,113113,17636159,5.953185,22.799764,6


,url,bytes,decompressed_bytes,sha256,etag
0,https://s3.amazonaws.com/logan-pub/c/DRR001152...,1001246,3471720,aaf4a077615323d01b6b7b00db557f0f8a85875a6112c1...,2aab8b332e12f06288bf4309c6596d09
1,https://s3.amazonaws.com/logan-pub/c/DRR000584...,1016976,3559616,6a51f077d5c7fdd5ec49e6d4601a9a39ea58422e260be8...,29a29c86a0d1d05c56810a5e3e054902
2,https://s3.amazonaws.com/logan-pub/c/DRR001026...,1038706,3687632,52a1de95f1919c2838ebd68dec4686eea62f599a3e7e13...,96ed72a46532fa981c69efca5c49d67e
3,https://s3.amazonaws.com/logan-pub/c/DRR001104...,1048790,4078810,dca4619bc6c20dfa0cce017d0ff70aa6fac670e9eac4ef...,e46a116077b6f9e3080fafbe358cf450
4,https://s3.amazonaws.com/logan-pub/c/DRR000185...,1057764,3684320,fdb534b078285671ea9e78a4615085255eeebe9bccd022...,d20179ab366a32c6e4086f435783b29a
5,https://s3.amazonaws.com/logan-pub/c/DRR001199...,1078885,5425187,7876022bbbf455437452a040ca26852ca55478c769d8f1...,c8ffae558f313ddee7675cd2be9ba657


In [4]:
display(metadata[['run_accession', 'scientific_name', 'library_source', 'library_strategy', 'instrument_platform']])

,run_accession,scientific_name,library_source,library_strategy,instrument_platform
0,DRR001152,Canis lupus familiaris,TRANSCRIPTOMIC,RNA-Seq,ILLUMINA
1,DRR000584,Botryococcus braunii,TRANSCRIPTOMIC,EST,LS454
2,DRR001026,Oryza sativa Japonica Group,TRANSCRIPTOMIC,RNA-Seq,ILLUMINA
3,DRR001104,Pinctada fucata,TRANSCRIPTOMIC,EST,LS454
4,DRR000185,Bacillus anthracis,GENOMIC,WGS,ILLUMINA
5,DRR001199,Mus musculus,TRANSCRIPTOMIC,AMPLICON,LS454


In [5]:
display(builds.sort_values(['repetition', 'variant']))
display(builds.groupby('variant')[['build_seconds', 'records_per_second', 'database_bytes']].agg(['median', 'min', 'max']))

,variant,build_seconds,records_per_second,database_bytes,page_count,page_size,integrity_check,repetition
1,metadata,1.011615,111814.318509,45793280,11180,4096,ok,1
0,no_metadata,0.962229,117553.066426,35106816,8571,4096,ok,1
2,metadata,0.997059,113446.672530,45793280,11180,4096,ok,2
3,no_metadata,0.769862,146926.392178,35106816,8571,4096,ok,2
5,metadata,1.137455,99443.909479,45793280,11180,4096,ok,3
4,no_metadata,0.882611,128157.240719,35106816,8571,4096,ok,3


build_seconds                     records_per_second  \
                   median       min       max             median   
variant                                                            
metadata         1.011615  0.997059  1.137455      111814.318509   
no_metadata      0.882611  0.769862  0.962229      128157.240719   

                                          database_bytes                      
                       min            max         median       min       max  
variant                                                                       
metadata      99443.909479  113446.672530     45793280.0  45793280  45793280  
no_metadata  117553.066426  146926.392178     35106816.0  35106816  35106816

In [6]:
assert validation['variants_identical'] is True
assert all(builds['integrity_check'] == 'ok')
validation

{'sequence_count': 113113,
 'total_bases': 17636159,
 'sequence_payload_sha256': '12eb1e3dce60e6a396f70d78b3e1324a8983b7a7c7ebca2a286932523dc1c2e7',
 'variants_identical': True,
 'accessions': ['DRR001152',
  'DRR000584',
  'DRR001026',
  'DRR001104',
  'DRR000185',
  'DRR001199'],
 'upstream_revisions': {'IndexThePlanet/Logan': 'b3da25311802544c4c0e53c44b98380acf4d04d6',
  'rchikhi_pasteur/logan-analysis': '5e92dafb87e40834ff2123d9b55e3bb27887fedb',
  'ababaian/petadex': 'a6bc326eb187925591b3988f57b4fadb1dba557d'},
 'python': '3.11.14 (main, Oct  9 2025, 16:16:55) [GCC 15.2.0]',
 'platform': 'Linux-6.18.49-x86_64-with-glibc2.40',
 'sqlite': '3.51.1'}

## Build interpretation

Database size and build time differences measure the cost of the complete metadata-enriched schema, including normalized run metadata, per-sequence abundance and header fields, and additional indexes. They do not measure the full Logan release, and they should not be extrapolated linearly to petabyte scale without additional distributed-system tests.